<font color="#CA0032"><h1 align="left">**Redes recurrentes profundas**</h1></font>

<font color="#6E6E6E"><h1 align="left">**Predicción de series temporales**</h1></font>

<h2 align="left">Manuel Sánchez-Montañés</h2>

<font color="#6E6E6E"><h2 align="left">manuel.smontanes@gmail.com</h2></font>

**Notebook: Manuel Sánchez-Montañés**

**Datos: Carlos Rosado**

### **Usaremos un esquema many to one:**

<img src="https://drive.google.com/uc?export=download&id=1iokh576AiK2iFhftPogSBsNXixAi-LBg" align="center" style="float" width="500">

In [ ]:
COLAB = True

## <font color="#CA3532"> **1. Importar librerías**

In [ ]:
import numpy as np
import pandas as pd

from keras.models import Sequential, load_model
from keras.layers import Dense, LSTM, GRU
from keras.callbacks import ModelCheckpoint

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score as R2_score

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import os

# fijo la semilla aleatoria por reproducibilidad
np.random.seed(7)

In [ ]:
def grafica_entrenamiento(tr_mse, val_mse):
    ax=plt.figure(figsize=(10,4)).gca()
    plt.plot(1+np.arange(len(tr_mse)), tr_mse)
    plt.plot(1+np.arange(len(val_mse)), val_mse)
    plt.title('mse del modelo', fontsize=18)
    plt.xlabel('epoca', fontsize=18)
    plt.ylabel('mse', fontsize=18)
    plt.legend(['entrenamiento', 'validación'], loc='upper left')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.show()

In [ ]:
def download_file_from_google_drive(file_id, dest_file, unzip=False):
  aux = "'https://drive.usercontent.google.com/download?id={}&export=download&confirm=t&uuid=9699f0e2-e760-49fc-b12e-49f140095280'".format(file_id)
  !wget $aux -O $dest_file
  if unzip:
    !unzip -qq -o $dest_file
    !rm $dest_file

## <font color="#CA3532"> **2. Carga de datos**

In [ ]:
!ls

In [ ]:
if COLAB:
    download_file_from_google_drive(file_id='12QZpA_L1JncFIVEryee0aeL66Ep3xpOy', dest_file='./datos_pasajeros.csv')

data = pd.read_csv('datos_pasajeros.csv', parse_dates=["fecha"])
data.head(20)

In [ ]:
data.tail()

In [ ]:
data.shape

In [ ]:
data.head(10)

In [ ]:
# tam ventana = 3

# primera ventana: predecir y=1125 en función de X = [[nan], [nan], [nan]]
# shape de esa primera ventana: (3,1)

# segunda ventana: predecir y=3592 en función de X = [[nan], [nan], [1125]]
# shape de esa primera ventana: (3,1)

# tercera ventana: predecir y=3001 en función de X = [[nan], [1125], [3592]]
# shape de esa primera ventana: (3,1)

# cuarta ventana: predecir y=2260 en función de X = [[1125], [3592], [3001]]
# shape de esa primera ventana: (3,1)

# quinta ventana: predecir y=2767 en función de X = [[3592], [3001], [2260]]
# shape de esa primera ventana: (3,1)

# Si solo tengo esas cinco ventanas:

# y.shape = (5,)
# X.shape = (5,3,1)

In [ ]:
!ls -la

In [ ]:
data.shape

In [ ]:
data["fecha"].diff().value_counts()

In [ ]:
data.shape

In [ ]:
data["fecha"].nunique()

## <font color="#CA3532"> **3. Preprocesado inicial y visualización de datos**

In [ ]:
data.dtypes

In [ ]:
fechas = data["fecha"].values
target = data["npasajeros"].values

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(fechas, target)
plt.title("Número de pasajeros (acumulado en la franja de tarde)", fontsize=20);

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(fechas[28:2*28], target[28:2*28], "o-")
plt.title("Número de pasajeros (acumulado en la franja de tarde)", fontsize=20)
plt.grid();

In [ ]:
plt.hist(target, bins=20);

## <font color="#CA3532"> **4. Transformación de la variable a predecir**

In [ ]:
# Transformación de escala (ajustar el factor en función del problema)
def transform(x):
    return x/5000
def inverse_transform(x_escalado):
    return x_escalado*5000

In [ ]:
transform(np.array([1,2,3]))

In [ ]:
inverse_transform(transform(np.array([1,2,3])))

## <font color="#CA3532"> **5. Enventanado de datos**

In [ ]:
if COLAB:
    download_file_from_google_drive(file_id='1LYuVxpFdsoxgl89tku6BtEH3HuYcGd2g',
                                    dest_file='./my_utils_series_temporales.py.zip', unzip=True)

from my_utils_series_temporales import int2dummy, enventanar, info_enventanado, NAN

In [ ]:
target_transf = transform(target)
target_transf[:5]

In [ ]:
# según mi librería:
series = [target_transf] # series a enventanar
se_saben_antes = [False] # adelanto o no las series
nombres_series = ["target_transf"]

print(np.shape(series))
print(np.shape(se_saben_antes))

In [ ]:
lookback = 14 # tamaño ventana: lookback es otro sinónimo de W_in

# target=0 en siguiente línea quiere decir cuál es el índice del target en objeto "series":
X, y = enventanar(series, target=0, se_saben_antes=se_saben_antes,
                  W_in=lookback)

print(X.shape, np.shape(y))

In [ ]:
X[0]

In [ ]:
y[0]

In [ ]:
X[1]

In [ ]:
y[1]

In [ ]:
X[3]

In [ ]:
y[3]

In [ ]:
series[0][:10]

In [ ]:
X[5]

In [ ]:
y[5]

In [ ]:
fechas_aux = [str(x).split("T")[0] for x in fechas]

info_enventanado(X[:10],y[:10],
                 nombres_series=nombres_series,
                 nombre_target="target",
                 tiempos=fechas_aux)

print(X.shape)
print(np.shape(target))

## <font color="#CA3532"> **6. Separación training-test**

In [ ]:
train_perc = 0.8
punto_corte = int(len(target)*train_perc)
punto_corte

In [ ]:
X_train = X[lookback:punto_corte]
y_train = y[lookback:punto_corte]
target_train = target[lookback:punto_corte]
fechas_train = fechas[lookback:punto_corte]

In [ ]:
X_test = X[punto_corte:]
y_test = y[punto_corte:]
target_test = target[punto_corte:]
fechas_test = fechas[punto_corte:]

In [ ]:
X_train.shape, y_train.shape, target_train.shape, fechas_train.shape

In [ ]:
X_test.shape, y_test.shape, target_test.shape, fechas_test.shape

## <font color="#CA3532"> **7. Construcción del modelo con Keras**

In [ ]:
model = Sequential()
model.add(GRU(2 , input_shape=(lookback, 1)))
model.add(Dense(1))
model.compile(loss="mean_squared_error", optimizer="rmsprop")

In [ ]:
model.summary()

In [ ]:
epochs = 500
Nval = 200
#batch_size = len(X_train[:-Nval]) # 128
batch_size = 32

control_sobreajuste_val = True

# Ahora train lo divido en tr (con el que realmente entreno)
# & val (con el que valido)
X_tr  = X_train[:-Nval] # quitando los 200 últimos valores
y_tr  = y_train[:-Nval]
X_val = X_train[-Nval:] # los 200 últimos valores
y_val = y_train[-Nval:]

if not control_sobreajuste_val:
    history = model.fit(X_train, y_train, epochs=epochs,
                        batch_size=batch_size, verbose=2)
else:
    acum_tr_mse = []
    acum_val_mse = []
    modelpath="model_current_best.h5"
    checkpoint = ModelCheckpoint(modelpath, monitor='val_loss', verbose=2, # val_mean_squared_error
                                 save_best_only=True,
                                 mode='min') # graba sólo los que mejoran en validación

    callbacks_list = [checkpoint]

    for e in range(epochs):
        history = model.fit(X_tr, y_tr,
                            batch_size=batch_size,
                            epochs=1,
                            callbacks=callbacks_list,
                            verbose=0,
                            validation_data=(X_val, y_val))

        acum_tr_mse  += history.history['loss'] # mean_squared_error
        acum_val_mse += history.history['val_loss'] # val_mean_squared_error

        if (e+1)%50 == 0:
            grafica_entrenamiento(acum_tr_mse, acum_val_mse)

model = load_model(modelpath) # recupero el mejor modelo en validación

## <font color="#CA3532"> **8. Chequeo del modelo (predicción a un día)**

In [ ]:
y_train_prediction = model.predict(X_train, verbose=0).flatten()
target_train_pred = inverse_transform(y_train_prediction)

y_test_prediction  = model.predict(X_test, verbose=0).flatten()
target_test_pred = inverse_transform(y_test_prediction)

In [ ]:
plt.figure(figsize=(15,7))
plt.plot(fechas_train, target_train, "--", c="royalblue", label="training (real)")
plt.plot(fechas_train, target_train_pred, "--", c="darkorange", label="training (pred a un día)")
plt.plot(fechas_test, target_test, "--", c="green", label="test (real)")
plt.plot(fechas_test, target_test_pred, "--", c="red", label="test (pred a un día)")
plt.legend()
plt.grid();

In [ ]:
plt.figure(figsize=(15,7))
plt.plot(fechas_train, target_train, "--", c="royalblue", label="training (real)")
plt.plot(fechas_train, target_train_pred, "--", c="darkorange", label="training (pred a un día)")
plt.plot(fechas_test, target_test, "--o", c="green", label="test (real)")
plt.plot(fechas_test, target_test_pred, "--o", c="red", label="test (pred a un día)")
plt.legend()
plt.xlim([fechas_test[-28*2], fechas_test[-1]])
plt.grid();

In [ ]:
np.sqrt(((target_test_pred - target_test)**2).mean())

In [ ]:
np.sqrt(((target_test[:-1] - target_test[1:])**2).mean())

In [ ]:
R2_score(target_test, target_test_pred)

In [ ]:
R2_score(target_test[1:], target_test[:-1])

## <font color="#CA3532"> **9. Predicciones a varios días en test mediante un modelo auto-regresivo**

In [ ]:
#1000, 1200, 1300 -> 1400
#1200, 1300, 1400 -> 1500
#1300, 1400, 1500

In [ ]:
def step_autoregresivo(model, ventana, X, i):
    # ventana: va a tener dimensiones (1, lookback, n_variables)
    # El target está en (0,:,0)
    # La posición (0,0,0) es el target en el día más antiguo de la ventana
    # La posición (0,0,-1) es el target en el día más reciente de la ventana
    # Las variables calendario están en (0,:,1:)
    # Las variables calendario para el último día de la ventana están en (0,-1,1:)

    z = model.predict(ventana, verbose=0)[0,0] # predicción nueva
    target_pred = inverse_transform(z)
    ventana_nueva = None
    if i < (len(X)-1):
        ventana_nueva = np.zeros(ventana.shape)
        # muevo un día hacia atrás todas las variables de la ventana:
        ventana_nueva[0,:-1,:] = ventana[0,1:,:].copy()

        # introduzco las variables calendario para el último día de la ventana:
        ventana_nueva[0,-1,1:] = X[i+1,-1,1:]

        # meto la predicción como última obervación del target:
        ventana_nueva[0,-1,0] = z


    return target_pred, ventana_nueva

In [ ]:
def autoregresivo(model, X):
    ventana = np.array([X[0]]).copy()
    salidas = []
    for i in range(len(X)):
        target_pred, ventana_nueva = step_autoregresivo(model, ventana, X, i)
        salidas.append(target_pred)
        ventana = ventana_nueva

    return salidas

In [ ]:
sal = autoregresivo(model, X_test)

In [ ]:
fig = plt.figure(figsize=(10,3))
ax = fig.add_subplot(1,1,1)
ax.plot(fechas_test, target_test, '--g', label='test')
plt.plot(fechas_test, sal, 'r', label='predicciones en test')
plt.title('test (predicción a varios días)')
plt.legend();